# Fase 2: Modelação e Demand Sensing
Nesta fase, transformamos os dados consolidados em inteligência preditiva. O foco é utilizar o comportamento digital (cliques) em conjunto com o histórico para antecipar as necessidades de inventário e mitigar as "ruturas virtuais" identificadas na fase anterior.

**Objetivos:**
1. **Feature Engineering:** Criar "memória" no modelo através de atributos temporais (*Lags*) e médias móveis, permitindo que o modelo aprenda padrões.
2. **Estabelecer um Baseline (Modelo Simples):** Criar um modelo básico (assumir que a procura de amanhã será igual à de hoje) para quantificar o desempenho da atual gestão reativa da empresa.
3. **Treino e Backtesting Preditivo:** Treinar um modelo de **Random Forest Regressor** e comparar a sua precisão face ao modelo mais simples através de uma divisão temporal (*Time-Series Split*). Utilizaremos a última semana da nossa janela temporal como conjunto de teste para provar a viabilidade em cenários reais.

In [1]:
!pip install -q scikit-learn matplotlib numpy


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Carregar o dataset consolidado da Fase 1
df = pd.read_csv('../dados/dataset_final_modelacao.csv')

# 2. Tipagem e Ordenação
df['order_date'] = pd.to_datetime(df['order_date'])
df = df.sort_values(by=['Product Name', 'order_date']).reset_index(drop=True)

print("✅ Dados carregados e ordenados.")
print(f"Produtos para modelação: {df['Product Name'].nunique()}")

# 3. Visualização rápida
display(df.head())

✅ Dados carregados e ordenados.
Produtos para modelação: 5


,order_date,Product Name,Order Item Quantity,clicks
0,2017-09-01,Field & Stream Sportsman 16 Gun Fire Safe,11,95.0
1,2017-09-02,Field & Stream Sportsman 16 Gun Fire Safe,12,106.0
2,2017-09-03,Field & Stream Sportsman 16 Gun Fire Safe,17,74.0
3,2017-09-04,Field & Stream Sportsman 16 Gun Fire Safe,14,76.0
4,2017-09-05,Field & Stream Sportsman 16 Gun Fire Safe,18,84.0


### 1. Feature Engineering
Os modelos de Machine Learning não percebem como funciona a passagem do tempo. Para resolver isto, criei **Lags** (atrasos temporais).

* **clicks_lag1, 2, 3:** Representam o volume de cliques nos 3 dias anteriores. É o sinal de "intenção de compra" que o cliente poderá estar a dar antes de efetivar a transação.
* **sales_lag1:** A venda do dia anterior, que serve como base para a inércia da procura.
* **vendas_media_3d:** Uma média móvel para perceber a tendência da semana e suavizar variações atípicas.

In [3]:
# 1. Garantir ordenação cronológica por produto
df = df.sort_values(by=['Product Name', 'order_date'])

# 2. Criação de Lags de Cliques (Sinal Digital)
for i in range(1, 4):
    df[f'clicks_lag{i}'] = df.groupby('Product Name')['clicks'].shift(i)

# 3. Criação de Lag de Vendas e Média Móvel (usamos uma janela de 3 dias para a média móvel)
df['sales_lag1'] = df.groupby('Product Name')['Order Item Quantity'].shift(1)
df['vendas_media_3d'] = df.groupby('Product Name')['Order Item Quantity'].transform(lambda x: x.rolling(window=3).mean())

# 4. Limpeza de Nulos (Início da série temporal)
# Removemos as linhas que não têm histórico (primeiros 3 dias de cada produto)
df_modelo = df.dropna().copy()

print(f"Amostras restantes para treino/teste: {len(df_modelo)}")
display(df_modelo.head(5))

Amostras restantes para treino/teste: 145


,order_date,Product Name,Order Item Quantity,clicks,clicks_lag1,clicks_lag2,clicks_lag3,sales_lag1,vendas_media_3d
3,2017-09-04,Field & Stream Sportsman 16 Gun Fire Safe,14,76.0,74.0,106.0,95.0,17.0,14.333333
4,2017-09-05,Field & Stream Sportsman 16 Gun Fire Safe,18,84.0,76.0,74.0,106.0,14.0,16.333333
5,2017-09-06,Field & Stream Sportsman 16 Gun Fire Safe,14,82.0,84.0,76.0,74.0,18.0,15.333333
6,2017-09-07,Field & Stream Sportsman 16 Gun Fire Safe,12,83.0,82.0,84.0,76.0,14.0,14.666667
7,2017-09-08,Field & Stream Sportsman 16 Gun Fire Safe,11,104.0,83.0,82.0,84.0,12.0,12.333333


### 2. Linha de Base Estatística (Baseline Simples)
Em modelação preditiva, antes de avaliarmos um modelo complexo, é mandatório estabelecer um *Baseline* simples (uma heurística de controlo). 

O objetivo não é adivinhar a política de *stock* atual da empresa, mas sim criar o modelo de previsão mais rudimentar possível: o modelo de inércia ou Random (assumir que as vendas de amanhã repetem as vendas de hoje). Ao quantificarmos o erro deste modelo básico, definimos a "fasquia" matemática mínima. O nosso modelo de Random Forest terá obrigatoriamente de superar esta métrica para provar que a introdução de dados web (cliques) e vendas médias móveis gera verdadeiro valor preditivo.

In [4]:
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Definir a realidade vs. a previsão heurística (Modelo Simples)
# A realidade: quantidade efetivamente vendida
y_real = df_modelo['Order Item Quantity']

# O Baseline: modelo simples (Previsão de hoje = Vendas de ontem)
y_simples_pred = df_modelo['sales_lag1']

# 2. Calcular as métricas da Linha de Base
mae_simples = mean_absolute_error(y_real, y_simples_pred)
r2_simples = r2_score(y_real, y_simples_pred)

# 3. Diagnóstico do Baseline
print("-- BASELINE (MODELO SIMPLES) --")
print(f"Erro Médio Absoluto (MAE): {mae_simples:.2f} unidades")
print(f"Precisão Preditiva (R² Score): {r2_simples:.4f}")

print("Objetivo: O modelo Random Forest terá de justificar a sua complexidade superando esta métrica básica através da aprendizagem de padrões.")


-- BASELINE (MODELO SIMPLES) --
Erro Médio Absoluto (MAE): 11.97 unidades
Precisão Preditiva (R² Score): 0.3774
Objetivo: O modelo Random Forest terá de justificar a sua complexidade superando esta métrica básica através da aprendizagem de padrões.


### Interpretação dos Resultados da Baseline
A avaliação do Modelo Simples (inércia) reflete as limitações da gestão reativa e estabelece o ponto de partida para a minha modelação:

1. **Baixo Poder Explicativo (R² de 37.7%):** Assumir que a procura de amanhã repete a de hoje apenas consegue explicar cerca de 38% da variabilidade real das vendas. Isto prova que o mercado é volátil e que mais de 60% das oscilações da procura ocorrem fora do padrão inercial.
2. **Impacto do Erro (MAE de ~12 unidades):** A heurística falha, em média, por 12 unidades diárias por produto. Num ambiente de retalho de alto volume, este desvio acumula-se rapidamente, justificando as crónicas ruturas de *stock* observadas na Fase 1.
3. **Justificação para Machine Learning:** A incapacidade do modelo simples de lidar com a volatilidade torna imperativo o uso de algoritmos não-lineares. O objetivo do **Random Forest** será cruzar este histórico de vendas com os sinais de intenção digital (*cliques* e *lags*) e vendas anteriores, procurando capturar a variância que a estatística básica ignora e elevar substancialmente a precisão preditiva.

## 3. Divisão Treino e Teste

Para validar a eficácia do modelo, apliquei uma divisão temporal. O modelo será treinado com os dados iniciais e testado numa janela "invisível" (a última semana de dados), simulando um cenário real de operação.

Conjunto de Treino: Período de 04/09 a 25/09 (Onde o modelo aprende os padrões).

Conjunto de Teste: Período de 26/09 a 02/10 (Onde validamos se o modelo previu corretamente as vendas).

In [5]:
# 1. Definir as colunas de entrada (Features) e a coluna de saída (Target)
features = ['clicks_lag1', 'clicks_lag2', 'clicks_lag3', 'sales_lag1', 'vendas_media_3d']
target = 'Order Item Quantity'

# 2. Definir a data de corte (Última semana para teste)
data_corte = '2017-09-26'

# 3. Criar os subconjuntos
train_data = df_modelo[df_modelo['order_date'] < data_corte].copy()
test_data = df_modelo[df_modelo['order_date'] >= data_corte].copy()

# 4. Separar X (entrada) e y (saída)
X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print(f"Treino: {X_train.shape[0]} amostras (Até {train_data['order_date'].max().date()})")
print(f"Teste: {X_test.shape[0]} amostras (A partir de {test_data['order_date'].min().date()})")

Treino: 110 amostras (Até 2017-09-25)
Teste: 35 amostras (A partir de 2017-09-26)


## 4. Modelação Preditiva com Random Forest

Com a infraestrutura de dados pronta e a Baseline estabelecida, avancei para a construção do modelo preditivo. Optei pelo algoritmo Random Forest Regressor pela sua robustez e capacidade de detetar padrões não-lineares — uma característica vital, dado que o nosso Modelo Simples (inércia) provou ser insuficiente para capturar a volatilidade real da procura.

### Estratégia de Treino e Validação:

1. **Features:** O modelo baseia-se na antecipação digital (lags de cliques), na inércia de mercado (venda do dia anterior) e na tendência de curto prazo (média móvel de 3 dias).
2. **Janela de Treino:** Focamos a aprendizagem no período de 04/09 a 25/09, garantindo que o modelo estuda apenas padrões reais e estabilizados (após o cálculo inicial das médias móveis).
3. **Método de Backtesting:** O modelo será avaliado em dois ambientes: no conjunto de treino e na última semana da janela temporal (26/09 a 02/10). Analisar o erro no Treino vs. Teste garante que não existe *overfitting*. O teste final validará se o modelo consegue bater a métrica de inércia.

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Instanciar o modelo
# Usei parâmetros conservadores para evitar o "overfitting" devido ao dataset pequeno
rf_model = RandomForestRegressor(
    n_estimators=100, 
    max_depth=5, 
    random_state=42
)

# 2. Treinar o modelo
rf_model.fit(X_train, y_train)

# 3. Gerar previsões para o conjunto de treino e teste
y_pred_train = rf_model.predict(X_train)
y_pred_test = rf_model.predict(X_test)

# 4. Cálculo de Métricas de Erro
mae_train = mean_absolute_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

mae_test = mean_absolute_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print("--- AVALIAÇÃO DO MODELO RANDOM FOREST ---")
print("1. DADOS DE TREINO:")
print(f"   Erro Médio Absoluto (MAE): {mae_train:.2f} unidades")
print(f"   Precisão (R² Score): {r2_train:.4f}")

print("\n2. DADOS DE TESTE:")
print(f"   Erro Médio Absoluto (MAE): {mae_test:.2f} unidades")
print(f"   Precisão (R² Score): {r2_test:.4f}")

# 5. Importância das Variáveis
importancia = pd.DataFrame({
    'Atributo': features,
    'Importância': rf_model.feature_importances_
}).sort_values(by='Importância', ascending=False)

print("\n--- FEATURE IMPORTANCE ---")
print(importancia.to_string(index=False))

--- AVALIAÇÃO DO MODELO RANDOM FOREST ---
1. DADOS DE TREINO:
   Erro Médio Absoluto (MAE): 4.16 unidades
   Precisão (R² Score): 0.9349

2. DADOS DE TESTE:
   Erro Médio Absoluto (MAE): 8.91 unidades
   Precisão (R² Score): 0.7218

--- FEATURE IMPORTANCE ---
       Atributo  Importância
vendas_media_3d     0.887607
    clicks_lag2     0.040983
     sales_lag1     0.040181
    clicks_lag3     0.017610
    clicks_lag1     0.013620


### 4.1. Interpretação dos Resultados

Os resultados do Random Forest revelam o verdadeiro impacto preditivo da integração de sinais digitais face à gestão inercial da empresa:

1. **Treino vs. Teste:** Como é característico nos modelos baseados em árvores de decisão, existe uma quebra entre o ambiente de treino (93.4%) e o ambiente de teste (72.1%). Contudo, esta variância não configura um *overfitting* destrutivo, pois o modelo retém um elevado poder preditivo em dados totalmente "invisíveis", provando que aprendeu padrões reais e não apenas ruído.
2. **Comparação com Baseline Simples:** O nosso objetivo era bater o Modelo Simples. O Random Forest conseguiu elevar a precisão preditiva (R²) de **37.7% para 72.18%**. Quase duplicámos a capacidade de prever a realidade.
3. **Impacto Logístico Direto (Redução do MAE):** No Modelo Simples, a empresa errava a sua previsão em cerca de 12 unidades por dia. O meu modelo reduziu este desvio para **8.91 unidades**. Esta redução do erro absoluto é a justificação matemática para a diminuição substancial de ruturas de *stock* que observaremos na Simulação de Negócio (Fase 3).

In [7]:
# GUARDAR PREVISÕES PARA A SIMULAÇÃO

# 1. Criar a tabela com a Realidade vs Previsão (usando a janela de teste)
resultados_ia = test_data[['order_date', 'Product Name', 'Order Item Quantity']].copy()
resultados_ia['Previsao_IA'] = y_pred_test

resultados_ia.to_csv('../dados/previsoes_ia.csv', index=False)

print("✅ Previsões do modelo guardadas com sucesso. Pronto para a Simulação da Fase 3.")

✅ Previsões do modelo guardadas com sucesso. Pronto para a Simulação da Fase 3.
